<a href="https://colab.research.google.com/github/Suthirtha2004/ml-from-scratch/blob/main/Coffee_Roasting_TF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Implementing the Coffee Roast dataset using Tensorflow

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [2]:
import numpy as np

def generate_coffee_data(
    n_samples=200,
    temperature_range=(175, 260),
    duration_range=(8, 18)
):
    rng = np.random.default_rng(42)

    X = rng.random(n_samples * 2).reshape(-1, 2)

    # Temperature
    X[:, 0] = (
        X[:, 0] * (temperature_range[1] - temperature_range[0])
        + temperature_range[0]
    )

    # Roasting duration
    X[:, 1] = (
        X[:, 1] * (duration_range[1] - duration_range[0])
        + duration_range[0]
    )

    Y = np.zeros(len(X))

    for i, (temperature, duration) in enumerate(X):

        # Good roasting requires both sufficient temperature
        # and an appropriate roasting duration.
        if 190 <= temperature <= 240 and 10 <= duration <= 15:
            Y[i] = 1

    return X, Y.reshape(-1, 1)

In [3]:
X,Y = generate_coffee_data()
print(X.shape,Y.shape)

(200, 2) (200, 1)


We need to normalize the values as its essential
1.First we need to add a normalization layer
2.Next adapt the data to learn the mean and variance

In [5]:
print(f"Temperature Max and Min values : {np.max(X[:,0]):0.2f}, {np.min(X[:,0]):0.2f}")
print(f"Duration Max and Min values : {np.max(X[:,1]):0.2f}, {np.min(X[:,1]):0.2f}")

norm_layer = tf.keras.layers.Normalization(axis=1)
norm_layer.adapt(X)
Xn = norm_layer(X)

print(f"Temp after normalization {np.max(Xn[:,0]):0.2f},{np.min(Xn[:,0]):0.2f}")
print(f"Duration after normalization {np.max(Xn[:,1]):0.2f},{np.min(Xn[:,1]):0.2f}")

Temperature Max and Min values : 258.26, 176.05
Duration Max and Min values : 17.92, 8.07
Temp after normalization 1.74,-1.65
Duration after normalization 1.70,-1.77


In [6]:
Xt = np.tile(Xn,(1000,1))
Yt = np.tile(Y,(1000,1))
print(Xt.shape,Yt.shape)

(200000, 2) (200000, 1)


Basic Tensorflow Model Implementation

In [8]:
tf.random.set_seed(1234)

model = Sequential([
    tf.keras.Input(shape=(2,)),
    Dense(3,activation = 'sigmoid',name='layer1'),
    Dense(1,activation='sigmoid',name='layer2')
])

# The Input(Shape) specifies the shape of the input and allows Tf to size the weights ans
# biases accordingly.

In [9]:
model.summary()

#L1 paramters = 2(features) * 3 (weights) + 3 (biases) for 3 neurons
#L2 parameters = 2(features) * 1(weight) + 1(bias)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ layer1 (Dense)                  │ (None, 3)              │             9 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer2 (Dense)                  │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13 (52.00 B)

 Trainable params: 13 (52.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
# Checking the weight and bias shape

W1,b1 = model.get_layer("layer1").get_weights()
W2,b2 = model.get_layer("layer2").get_weights()

print(f"W1 : {W1.shape}:\n" ,W1, f"\nb1 : {b1.shape} : ",b1)
print(f"W2 : {W2.shape}:\n" ,W2, f"\nb2 : {b2.shape} : ",b2)

W1 : (2, 3):
 [[ 0.2650411  -0.5980329   0.7096797 ]
 [-0.47601426 -0.65162325  1.050271  ]] 
b1 : (3,) :  [0. 0. 0.]
W2 : (3, 1):
 [[-0.57899576]
 [-0.22798985]
 [-0.22700346]] 
b2 : (1,) :  [0.]


In [12]:
# model.compile - defines the loss function and specifies the compile optimization
# model.fit - runs gradient descent and fits the weights into the data

model.compile(
    loss = tf.keras.losses.BinaryCrossentropy(),
    optimizer = tf.keras.optimizers.Adam(learning_rate = 0.01),
)

model.fit(
    Xt,Yt,
    epochs = 10
)

# Epoch means how many times the total training set will be applied and for the effiency
# the training data is divided into batched and the default size of batch in Tf is 32
# Here we have 200000 and so the expanded batch size of 32

Epoch 1/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - loss: 0.1704
Epoch 2/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - loss: 0.0858
Epoch 3/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - loss: 0.0815
Epoch 4/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.0788
Epoch 5/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.0753
Epoch 6/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.0712
Epoch 7/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step - loss: 0.0685
Epoch 8/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.0662
Epoch 9/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step - loss: 0.0633
Epoch 10/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.0606


In [13]:
#Predictions - Once trained we have to recall the model to make prediction and the shape of the test data should be (m,2)
# and also be normalised

X_test = np.array([
    [200,13.9],
    [200,17]
])

X_testn = norm_layer(X_test)
predictions = model.predict(X_testn)
print(f"Predictions = \n" , predictions)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
Predictions = 
 [[0.95331365]
 [0.00184491]]


In [14]:
# We have got the values in probability

ycap = np.zeros_like(predictions)
for i in range(len(predictions)):
  if predictions[i] >= 0.5:
    ycap[i] = 1
  else:
    ycap[i] = 0

print(f"Output :\n {ycap}")


Output :
 [[1.]
 [0.]]
